In [ ]:
import torch

torch.cuda.empty_cache()

In [ ]:
LR = 5e-4
BATCH_SIZE = 64
RESULTS_DIR_BASE_NAME = "ablation-prototype"
MAX_STEPS = 4000
EVAL_STEPS = 500
PREDICTION_LENGTH = 128
FULL_SUBSAMPLING = False
STRIDE = 58

In [ ]:
from fusiontimeseries.lib.config import FTSConfig

fts_config = FTSConfig(
    context_length=256,
    prediction_length=PREDICTION_LENGTH,
    sampling_stride=STRIDE,
    pred_tail_timestamps=80,
    batch_size=BATCH_SIZE,
    learning_rate=LR,
    lr_scheduler_type="linear",
    optimizer_type="adamw_torch_fused",
    max_grad_norm=1.0,
    max_steps=MAX_STEPS,
    eval_steps=EVAL_STEPS,
    gradient_accumulation_steps=1,
    full_subsampling=FULL_SUBSAMPLING,
    padding_value=0.0,  # chronos2 has NaN as padding value
    padding_mask_default=0.0,
    padding_mask_indicator=1.0,
    sampling_strategy="tail_mean",
    sampling_bins=5,
)

In [ ]:
from torch.utils.data import Dataset

from fusiontimeseries.ablations.dataset import FTSDataBenchmarkingMixin
from fusiontimeseries.lib.dataset import FTSDataProcessingMixin

import numpy as np
from typing import Any


class BaselineTimeseriesDataset(
    Dataset, FTSDataProcessingMixin, FTSDataBenchmarkingMixin
):
    def __init__(
        self,
        time_series: list[np.ndarray],
        config: FTSConfig,
        *args: Any,
        **kwargs: Any,
    ) -> None:
        self.time_series = time_series
        self.config = config

        torch.manual_seed(self.config.random_seed)
        torch.cuda.manual_seed_all(self.config.random_seed)
        np.random.seed(self.config.random_seed)

    @staticmethod
    def collate_fn(
        batch: list[dict[str, torch.Tensor]],
    ) -> dict[str, list[torch.Tensor]]:
        collated_batch = {}
        for key in batch[0].keys():
            new_value = torch.cat([sample[key] for sample in batch])
            # print(f"Collating key: {key}, shape: {new_value.shape}")
            collated_batch[key] = new_value
        return collated_batch

    def prepare_sample(self, idx: int) -> dict[str, torch.Tensor]:
        time_series: np.ndarray = self.time_series[idx]

        ########## Perform Log Scaling ##########
        # time_series = LogScaling.apply_log_scaling(time_series)
        #########################################

        L: int = len(time_series)

        contexts: list[torch.Tensor] = []
        context_masks: list[torch.Tensor] = []
        future_targets: list[torch.Tensor] = []
        freqs: list[torch.Tensor] = []

        for cutoff_idx in range(
            80,
            L - self.config.prediction_length + 1,
            self.config.sampling_stride or 1,
        ):
            ############# Context ##############
            history: np.ndarray = time_series[:cutoff_idx, ...]
            context_start_idx = self.config.context_length - cutoff_idx

            context = torch.full(
                size=(self.config.context_length,),
                fill_value=self.config.padding_value,
                dtype=torch.float32,
            )
            context[context_start_idx:] = torch.Tensor(history)
            contexts.append(context)

            ############## Context mask ##############
            context_mask = torch.full_like(
                context,
                fill_value=self.config.padding_mask_default,
                dtype=torch.float32,
            )
            # assign padding indicator to padded positions (opposite to chronos2)
            context_mask[:context_start_idx] = self.config.padding_mask_indicator
            context_masks.append(context_mask)

            ############### Future ##############
            target: np.ndarray = time_series[
                cutoff_idx : cutoff_idx + self.config.prediction_length, ...
            ]
            future = torch.tensor(target, dtype=torch.float32)
            future_targets.append(future)

            ############## Frequency ##############
            freqs.append(torch.tensor([0.0], dtype=torch.long))

        return {
            "context": torch.stack(contexts),
            "context_mask": torch.stack(context_masks),
            "future_target": torch.stack(future_targets),
            "freq": torch.stack(freqs).to(torch.long),
        }

    def __len__(self) -> int:
        return len(self.time_series)

    def __getitem__(self, idx: int) -> dict[str, torch.Tensor]:
        return self.prepare_sample(idx)


TimeSeriesDataset = BaselineTimeseriesDataset

train_dataset, val_dataset = TimeSeriesDataset.train_val_split(fts_config)

- Goal is to reduce needed context
- mean over complete
- check different normalizations for zero-shot
- correlation between what values are in first mean-std-patch and how is performance on those fluxes? 
- table with difffernt normalizations
- val 13 is underreprensented in training data
- normalization auf zero-shot benchmark checken
- Bilinear Lora on best LoRA setting
- Check RMSE on average flux autregressive validation

In [ ]:
from timesfm import pytorch_patched_decoder as timesfm_lib


def _masked_mean_std(
    inputs: torch.Tensor, padding: torch.Tensor
) -> tuple[torch.Tensor, torch.Tensor]:
    """Calculates mean and standard deviation of `inputs` across axis 1.

    It excludes values where `padding` is 1.

    Args:
      inputs: A PyTorch tensor of shape [b, n, p].
      padding: A PyTorch tensor of shape [b, n, p] with values 0 or 1.

    Returns:
      A tuple containing the mean and standard deviation.
      We return the statistics of the first patch with more than three non-padded
      values.
    """
    arr = inputs
    pad = padding

    # Create a mask where padding is 0
    mask = 1 - pad

    masked_median = torch.median(arr[mask == 1])
    masked_median = masked_median.unsqueeze(0)
    masked_iqr = torch.quantile(arr[mask == 1], 0.75) - torch.quantile(
        arr[mask == 1], 0.25
    )
    masked_iqr = masked_iqr.unsqueeze(0)
    # Calculate the number of valid elements
    # num_valid_elements = torch.sum(mask, dim=1)
    # num_valid_elements = torch.where(
    #     num_valid_elements == 0,
    #     torch.tensor(1,
    #                  dtype=num_valid_elements.dtype,
    #                  device=num_valid_elements.device),
    #     num_valid_elements,
    # )

    # # Calculate the masked sum and squared sum
    # masked_sum = torch.sum(arr * mask, dim=1)
    # masked_squared_sum = torch.sum((arr * mask)**2, dim=1)

    # # Calculate the masked mean and standard deviation
    # masked_mean = masked_sum / num_valid_elements
    # masked_var = masked_squared_sum / num_valid_elements - masked_mean**2
    # masked_var = torch.where(
    #     masked_var < 0.0,
    #     torch.tensor(0.0, dtype=masked_var.dtype, device=masked_var.device),
    #     masked_var,
    # )
    # masked_std = torch.sqrt(masked_var)

    return masked_median, masked_iqr


timesfm_lib._masked_mean_std = _masked_mean_std

In [ ]:
from timesfm import TimesFmCheckpoint, TimesFmHparams
from timesfm.timesfm_torch import TimesFmTorch


def get_model():

    repo_id = "google/timesfm-2.0-500m-pytorch"

    hparams = TimesFmHparams(
        backend=fts_config.device,  # type: ignore
        per_core_batch_size=fts_config.batch_size,
        horizon_len=fts_config.prediction_length,
        context_len=fts_config.context_length,
        num_layers=50,
        use_positional_embedding=False,
    )

    tfm = TimesFmTorch(
        hparams=hparams, checkpoint=TimesFmCheckpoint(huggingface_repo_id=repo_id)
    )

    model: timesfm_lib.PatchedTimeSeriesDecoder | None = tfm._model

    if model is None:
        raise ValueError("Model is None")

    return model


model: timesfm_lib.PatchedTimeSeriesDecoder = get_model()
# horizon_len internally stays at 128 and we crop the prediction output to
#  avoid architectural mismatch and still use pretrained weights accordingly

In [ ]:
# patch config to be HF Trainer compatible

from timesfm.pytorch_patched_decoder import TimesFMConfig
from fusiontimeseries.ablations.trainer import patch_times_fm_config

patch_times_fm_config(TimesFMConfig)

In [ ]:
ts1 = train_dataset.time_series[1]
ts1.shape, ts1.min(), ts1.max()

In [ ]:
# scale ts1 using Robust Scaling (median + IQR)

median = np.median(ts1)
iqr = np.percentile(ts1, 75) - np.percentile(ts1, 25)
scaled_ts1 = (ts1 - median) / iqr
scaled_ts1.shape, scaled_ts1.min(), scaled_ts1.max(), iqr, median

In [ ]:
# over all timseries in the train dataset, compue the median and IQR and plot all medians and IQRs

medians = []
iqrs = []

for ts in train_dataset.time_series:
    medians.append(np.median(ts))
    iqrs.append(np.percentile(ts, 75) - np.percentile(ts, 25))

from matplotlib import pyplot as plt

plt.plot(medians, label="Medians")
plt.plot(iqrs, label="IQRs")

In [ ]:
from matplotlib import pyplot as plt

fig, axes = plt.subplots(2, 1, figsize=(12, 8))
axes[0].plot(ts1, label="Original Timeseries")
axes[0].set_title("Original Timeseries")
axes[1].plot(scaled_ts1, label="Robust Scaled Timeseries", color="orange")
axes[1].set_title("Robust Scaled Timeseries")

In [ ]:
test_sample = train_dataset.prepare_sample(0)

In [ ]:
input_ts: torch.Tensor = test_sample["context"][1].unsqueeze(0)
target_ts: torch.Tensor = test_sample["future_target"][1].unsqueeze(0)
input_padding: torch.Tensor = test_sample["context_mask"][1].unsqueeze(0)
freq: torch.Tensor = test_sample["freq"][1].unsqueeze(0)

In [ ]:
# inter-quartile range and median of input ts
input_ts_non_padded = input_ts[input_padding == 0.0]  # select non-padded values
input_median = torch.median(input_ts_non_padded)
input_iqr = torch.quantile(input_ts_non_padded, 0.75) - torch.quantile(
    input_ts_non_padded, 0.25
)
input_ts = (input_ts - input_median) / input_iqr
(
    input_ts.shape,
    input_ts.min(),
    input_ts.max(),
    input_iqr,
    input_median,
    input_padding.shape,
    input_padding.min(),
    input_padding.max(),
)

In [ ]:
bsize = input_ts.shape[0]
patched_inputs = input_ts.view(bsize, -1, 32)
patched_pads = input_padding.view(bsize, -1, 32)

patched_inputs = torch.where(
    torch.abs(patched_pads - 1.0) < 1e-6,
    torch.tensor(0.0, dtype=patched_inputs.dtype, device=patched_inputs.device),
    patched_inputs,
)

In [ ]:
patched_inputs.shape, patched_pads.shape

In [ ]:
# Selecting the first patch with more than 3 unpadded values.
unpadded_patch_sum = torch.sum(1 - patched_pads, dim=2)
unpadded_patch_sum.shape, unpadded_patch_sum

In [ ]:
indices = torch.argmax((unpadded_patch_sum >= 3).to(torch.int32), dim=1)
row_sum = (unpadded_patch_sum).to(torch.int32).sum(dim=1)
patch_indices = torch.where(row_sum == 0, unpadded_patch_sum.shape[1] - 1, indices)

indices, row_sum, patch_indices

In [ ]:
bidxs = torch.arange(patched_inputs.shape[0])
bidxs.shape, bidxs

In [ ]:
arr = patched_inputs[bidxs, patch_indices, :]
pad = patched_inputs[bidxs, patch_indices, :]
# Create a mask where padding is 0
mask = 1 - pad

masked_median = torch.median(patched_inputs[patched_pads == 0])
masked_median = masked_median.unsqueeze(0)
masked_iqr = torch.quantile(patched_inputs[patched_pads == 0], 0.75) - torch.quantile(
    patched_inputs[patched_pads == 0], 0.25
)
masked_iqr = masked_iqr.unsqueeze(0)

# Calculate the number of valid elements
num_valid_elements = torch.sum(mask, dim=1)
num_valid_elements = torch.where(
    num_valid_elements == 0,
    torch.tensor(1, dtype=num_valid_elements.dtype, device=num_valid_elements.device),
    num_valid_elements,
)

# Calculate the masked sum and squared sum
masked_sum = torch.sum(arr * mask, dim=1)
masked_squared_sum = torch.sum((arr * mask) ** 2, dim=1)

# Calculate the masked mean and standard deviation
masked_mean = masked_sum / num_valid_elements
masked_var = masked_squared_sum / num_valid_elements - masked_mean**2
masked_var = torch.where(
    masked_var < 0.0,
    torch.tensor(0.0, dtype=masked_var.dtype, device=masked_var.device),
    masked_var,
)
masked_std = torch.sqrt(masked_var)

In [ ]:
masked_mean, masked_std, masked_median, masked_iqr

In [ ]:
input_ts.shape

In [ ]:
b, p, _ = patched_inputs.shape
sub = masked_mean
div = masked_std

In [ ]:
norm_inputs = (patched_inputs - sub) / (div)
# norm_inputs = patched_inputs

In [ ]:
plt.plot(norm_inputs.view(b, -1)[0])

In [ ]:
new_norm_inputs = norm_inputs * (1.0 - patched_pads)
concat_inputs = torch.cat([new_norm_inputs, patched_pads], dim=-1)
concat_inputs.shape, concat_inputs.min(), concat_inputs.max()

In [ ]:
model_input = model.input_ff_layer(concat_inputs)
model_input.shape, model_input.min(), model_input.max()

In [ ]:
model_input[0].shape, model_input[0].min(), model_input[0].max()

In [ ]:
patched_padding = torch.min(patched_pads, dim=-1)[
    0
]  # Get the values from the min result
patched_padding.shape, patched_padding.min(), patched_padding.max()

In [ ]:
model_output = model.stacked_transformer(model_input, patched_padding)
model_output.shape, model_output.min(), model_output.max()

In [ ]:
model_output[0].shape, model_output[0].min(), model_output[0].max()

In [ ]:
output_ts = model.horizon_ff_layer(model_output)
output_ts.shape, output_ts.min(), output_ts.max()

In [ ]:
b, n, _ = output_ts.shape
final_output_ts = output_ts.view(b, n, model.config.horizon_len, 10)
final_output_ts.shape, final_output_ts.min(), final_output_ts.max()

In [ ]:
predictions = final_output_ts * masked_std.view(b, p, 1, 1) + masked_mean.view(
    b, p, 1, 1
)
predictions.shape, predictions.min(), predictions.max()

In [ ]:
# rescale by multiplying by IQR and adding median back
predictions = final_output_ts * input_iqr + input_median

In [ ]:
patch_idx = 7
pred = predictions[0, patch_idx, :, 0].detach().cpu().numpy()
prev = train_dataset.time_series[1][: patch_idx * 32 + 32]
plt.plot(np.concatenate([prev, pred]), label="Predicted Timeseries")
plt.vlines(
    len(prev),
    ymin=prev.min(),
    ymax=prev.max(),
    color="red",
    linestyle="--",
    label="Prediction Start",
)

In [ ]:
input_ts.shape, input_padding.shape

In [ ]:
predictions: torch.Tensor = model(
    input_ts=input_ts, input_padding=input_padding, freq=freq
)

In [ ]:
predictions.shape

In [ ]:
from matplotlib import pyplot as plt

patch_idx = 5
pred = predictions[0, patch_idx, :, 0].detach().cpu().numpy()
prev = train_dataset.time_series[0][: 138 - 32 - 32]
plt.plot(np.concatenate([prev, pred]), label="Predicted Timeseries")
plt.vlines(
    len(prev),
    ymin=prev.min(),
    ymax=prev.max(),
    color="red",
    linestyle="--",
    label="Prediction Start",
)

In [ ]:
from fusiontimeseries.lib.conditioning import ConditionRegistry


def calculate_loss(inputs: dict[str, torch.Tensor]) -> torch.Tensor:
    # Tensor[B, N]
    p_raw: torch.Tensor | None = inputs.pop(
        "operating_parameters", None
    )  # remove before forward, otherwise TypeError in Trainer

    input_ts: torch.Tensor = inputs["context"]
    target_ts: torch.Tensor = inputs["future_target"]
    input_padding: torch.Tensor = inputs["context_mask"]
    freq: torch.Tensor = inputs["freq"]

    with ConditionRegistry.patch(op_params=p_raw):
        # predictions shape: (batch_size, context_len // input_patch_size, prediction_length, mean + num_quantiles)
        predictions: torch.Tensor = model(
            input_ts=input_ts, input_padding=input_padding, freq=freq
        )

In [ ]:
from fusiontimeseries.loralib.layers import Linear

model = Linear.convert(
    module=model,
    kind="LoRA",
    lora_rank=8,
    lora_alpha=16,
    target_module_names=None,  # slap LoRA on all linear layers
)

In [ ]:
from pathlib import Path

from fusiontimeseries.lib.get_next_path import get_next_path

base_dir = Path("./results")
base_dir.mkdir(parents=True, exist_ok=True)
output_dir = get_next_path(base_fname=RESULTS_DIR_BASE_NAME, base_dir=base_dir)
output_dir.mkdir(parents=True, exist_ok=False)
print(f"Output directory created at: {output_dir}")

In [ ]:
from fusiontimeseries.loralib.utils import mark_only_lora_as_trainable
from fusiontimeseries.loralib.utils import print_trainable_parameters

mark_only_lora_as_trainable(model=model, bias="none")
print_trainable_parameters(model, save_path=output_dir / "trainable_params.json")

In [ ]:
from transformers.training_args import TrainingArguments

training_arguments = TrainingArguments(
    output_dir=str(output_dir),
    per_device_train_batch_size=fts_config.batch_size,
    per_device_eval_batch_size=fts_config.batch_size,
    learning_rate=fts_config.learning_rate,
    lr_scheduler_type=fts_config.lr_scheduler_type,
    optim=fts_config.optimizer_type,
    logging_strategy="steps",
    logging_steps=fts_config.eval_steps,
    disable_tqdm=False,
    report_to="none",
    max_steps=fts_config.max_steps,
    gradient_accumulation_steps=fts_config.gradient_accumulation_steps,
    dataloader_num_workers=0,
    tf32=False,
    bf16=False,
    save_only_model=True,
    prediction_loss_only=True,
    save_total_limit=2,
    save_strategy="steps",
    save_steps=fts_config.eval_steps,
    eval_strategy="steps",
    eval_steps=fts_config.eval_steps,
    load_best_model_at_end=False,  # keep last model since validation set is quite unexpressive
    metric_for_best_model="eval_loss",
    use_cpu=False,
    label_names=[
        "future_target"
    ],  # must be truthy for HF Trainer to use overridden compute_loss
    remove_unused_columns=False,  # needed to not accidentally remove columns that our custom compute_loss relies on
    max_grad_norm=fts_config.max_grad_norm,
)
training_arguments._n_gpu = 1

In [ ]:
import json

from fusiontimeseries.ablations.trainer import TimesFMTrainer

trainer = TimesFMTrainer(
    model=model,
    train_args=training_arguments,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    fts_config=fts_config,
    data_collator=TimeSeriesDataset.collate_fn,
)
with open(output_dir / "training_args.json", "w") as f:
    json.dump(trainer.args.to_dict(), f, indent=4)
fts_config.save_config(output_dir / "fts_config.json")

In [ ]:
import json
from fusiontimeseries.loralib.utils import lora_state_dict

train_output = trainer.train()

with open(output_dir / "train_summary.json", "w") as f:
    json.dump(train_output._asdict(), f, indent=4)

lora_weights = lora_state_dict(model)
torch.save(lora_weights, output_dir / "lora_weights.pt")

In [ ]:
benchmark_data = TimeSeriesDataset.get_benchmark_flux_traces(fts_config)
model = model.eval()

In [ ]:
from fusiontimeseries.lib.benchmarking import rmse_with_standard_error
from fusiontimeseries.lib.dataset import FluxData


def evaluate_model_on_test_data(
    model,
    config: FTSConfig,
    benchmark_data: dict[int, FluxData],
    start_context_length: int,
) -> dict[str, float | dict[int, list[float]]]:
    forecasts: dict[int, list[float]] = {}
    ground_truths: dict[int, list[float]] = {}
    for flux_id, flux_data in benchmark_data.items():
        flux_data: FluxData
        energy_flux = np.array(flux_data.energy_flux)
        min_energy_flux = np.min(energy_flux)
        energy_flux = np.log2(energy_flux + min_energy_flux + 1.0)
        op_params = (
            torch.Tensor(flux_data.operating_parameters).unsqueeze(0).to(config.device)
        )

        ctx: np.ndarray = energy_flux[:start_context_length]
        while len(ctx) < len(energy_flux):
            with torch.no_grad():
                context_start_idx = config.context_length - len(ctx)
                tctx = torch.full(
                    size=(1, config.context_length),
                    fill_value=config.padding_value,
                )
                tctx[0, context_start_idx:] = torch.tensor(ctx)
                context_mask = torch.full_like(
                    tctx, fill_value=config.padding_mask_default
                )  # 0.0
                # context_mask[:cutoff_idx] = self.config.padding_mask_indicator context_masks.append(context_mask)
                context_mask[0, :context_start_idx] = (
                    config.padding_mask_indicator
                )  # 1.0

                with ConditionRegistry.patch(op_params=op_params):
                    with torch.autocast(device_type=config.device, dtype=torch.float32):
                        # predictions shape: (batch_size, num_patches, horizon_len, num_quantiles + 1(mean))
                        predictions: torch.Tensor = model(
                            tctx.to(config.device, non_blocking=True),
                            context_mask.to(config.device, non_blocking=True).float(),
                            torch.tensor([[0.0]], dtype=torch.long).to(
                                config.device, non_blocking=True
                            ),
                        )
            forecast = predictions[0, -1, : config.prediction_length, 0].cpu().numpy()
            ctx = np.concatenate([ctx, forecast])

        forecasts[flux_id] = (
            np.exp2(ctx[: len(energy_flux)]) - min_energy_flux - 1.0
        ).tolist()
        ground_truths[flux_id] = flux_data.energy_flux

    benchmark_means: list[np.floating] = [
        np.mean(gt[-config.pred_tail_timestamps :]) for gt in ground_truths.values()
    ]
    forecast_means: list[np.floating] = [
        np.mean(fc[-config.pred_tail_timestamps :]) for fc in forecasts.values()
    ]
    rmse, rmse_se = rmse_with_standard_error(
        y_true=np.array(benchmark_means), y_pred=np.array(forecast_means)
    )
    return {
        "rmse": rmse,
        "rmse_standard_error": rmse_se,
        "forecasts": forecasts,
        "ground_truths": ground_truths,
    }

In [ ]:
id_results = evaluate_model_on_test_data(
    model=model,
    config=fts_config,
    benchmark_data=benchmark_data["id"],
    start_context_length=80,
)
id_results["rmse"], id_results["rmse_standard_error"]

In [ ]:
ood_results = evaluate_model_on_test_data(
    model=model,
    config=fts_config,
    benchmark_data=benchmark_data["ood"],
    start_context_length=80,
)
ood_results["rmse"], ood_results["rmse_standard_error"]

In [ ]:
import matplotlib.pyplot as plt

print(id_results["forecasts"].keys())

IDX = 115
plt.figure(figsize=(8, 6))
plt.plot(id_results["ground_truths"][IDX])
plt.plot(id_results["forecasts"][IDX])

# Ablation Studies

## Timeseries Augmentation

- Basic: LR 5e-4, effective bs 128, pred len 128, max steps 4000, eval steps 400, noticed: val set gets worse from the start onwards.
- Basic LoRA:                                                                                               15.54 +- 5.13; 7.69 +- 1.68
- Change: fully-subsample 241 training samples -> Train/Val split: 723 (prev. 241) / 3 time series.         16.36 +- 5.71; 3.94 +- 1.21
- Change: lower stride to 1 to have more samples -> problem only two timeseries in one batch (2 * 59)       17.11 +- 6.08; 5.41 +- 1.71
- Change: random cropping ->                                                                                18.47 +- 6.06; 4.14 +- 1.04

During evaluation on test set we also start from 80, two-split strategy starts from 80 (and 138), tailoring the model to give better forecasts from 80 onwards.

## Sampling Strategies

- Tail-mean sampling 14.26 +- 4.92; 7.57 +- 1.39

In [ ]:
id_results = TimeSeriesDataset.evaluate_model_on_test_data(
    model=model,
    config=fts_config,
    benchmark_data=benchmark_data["id"],
    start_context_length=202,
)

In [ ]:
import numpy as np


for forecast, target in zip(
    id_results["forecasts"].values(), id_results["ground_truths"].values()
):
    print(
        f"Forecast last 80 mean: {np.mean(forecast[-80:]):.4f}, Target last 80 mean: {np.mean(target[-80:]):.4f}"
    )

In [ ]:
len(id_results["forecasts"][115][:-80])

In [ ]:
id_results = evaluate_model_on_test_data(
    model=model,
    config=fts_config,
    benchmark_data=benchmark_data["id"],
    start_context_length=80,
)
id_results["rmse"], id_results["rmse_standard_error"]

In [ ]:
ood_results = evaluate_model_on_test_data(
    model=model,
    config=fts_config,
    benchmark_data=benchmark_data["ood"],
    start_context_length=80,
)
ood_results["rmse"], ood_results["rmse_standard_error"]

In [ ]:
# evaluate id and ood performance across different context lengths starting with 2, up to 202. Plot it

cl_rmse_results: list[tuple[int, float, float]] = []
for cl in range(22, 202 + 1, 20):
    id_results = evaluate_model_on_test_data(
        model=model,
        config=fts_config,
        benchmark_data=benchmark_data["id"],
        start_context_length=cl,
    )
    cl_rmse_results.append((cl, id_results["rmse"], id_results["rmse_standard_error"]))

In [ ]:
# save results to json
with open(output_dir / "cl_rmse_results.json", "w") as f:
    json.dump(
        [
            {"context_length": cl, "rmse": rmse, "rmse_standard_error": se}
            for cl, rmse, se in cl_rmse_results
        ],
        f,
        indent=4,
    )

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))

context_lengths = [x[0] for x in cl_rmse_results]
rmses = [x[1] for x in cl_rmse_results]
rmse_errors = [x[2] for x in cl_rmse_results]

plt.errorbar(
    context_lengths,
    rmses,
    yerr=rmse_errors,
    marker="o",
    capsize=2,
    alpha=1,
    color="pink",
    label="LoRA Fine-tuned",
)

plt.xticks(range(2, 203, 20))
plt.xlim(-12, 220)
plt.xlabel("Start Context Length")
plt.ylabel("RMSE ± Standard Error")
plt.title("ID Performance vs Context Length")
# plt.grid(axis="y")
# plt.legend()
plt.show()

# Hyperparameter seach (manual)

500 steps, no lr scheduler, LoRA on every Linear Layer in the network, Trainable parameters: 5,223,424 / 504,052,384 (1.04%), GPU RAM ~8/16GB

|Batch Size (observed) | Learning Rate | ID RMSE +- SE | OOD RMSE +- SE| Prediction Length |
|-----------|---------------|---------------|---------------|-------------------|
|64(128)   | 1e-5          | 36.13 +- 9.86 | 21.21 +- 3.78 | 128 (base) |
|64(128)   | 1e-4          | 25.91 +- 7.40 | 13.33 +- 2.67 | 128 (base) |
|64(128)   | 5e-4          | 22.56 +- 8.24 | 10.09 +- 2.18 | 128 (base) |
|64(128)   | 1e-3          | 23.38 +- 7.20 | 8.01 +- 2.21 | 128 (base) |
|128(256)   | 1e-4          | 24.51 +- 6.01 | 15.62 +- 4.55 | 128 (base) |
|128(256)   | 5e-5          | 25.44 +- 6.48 | 16.48 +- 4.22 | 128 (base) |
|64(192)   | 5e-4          | 23.55 +- 7.46 | 10.74 +- 2.54 | 64 |

- 64 pred len: Sample future timestamps are ['80 - 144', '141 - 205', '202 - 266'] = 3 samples per sample
- 128 pred len (original timesfm): Sample future timestamps are ['80 - 208', '138 - 266'] = 2 samples per sample